# Demografisk profil — Folkeregister

Analyserer kohorters utvikling: overlevelseskurver, giftermålsrater og medianalderens utvikling over simuleringsperioden.

**Kilde:** `folkeregister.cohort_demographics` (Gold-lag)  
**Relatert issue:** US E-3 / #95

In [ ]:
import os
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False})

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute("INSTALL delta;  LOAD delta;")

_raw = os.environ.get("MINIO_ENDPOINT", "http://minio.slettix-analytics.svc.cluster.local:9000")
_endpoint = _raw.replace("http://", "").replace("https://", "")

con.execute(f"""
    CREATE OR REPLACE SECRET minio (
        TYPE      S3,
        KEY_ID    '{os.environ.get("MINIO_ACCESS_KEY", "admin")}',
        SECRET    '{os.environ.get("MINIO_SECRET_KEY", "changeme")}',
        ENDPOINT  '{_endpoint}',
        URL_STYLE 'path',
        USE_SSL   false
    );
""")

COHORT_PATH = "s3://gold/folkeregister/cohort_demographics"
print("DuckDB klar.", duckdb.__version__)

In [ ]:
df = con.execute(f"SELECT * FROM delta_scan('{COHORT_PATH}')").df()
print(f"{len(df):,} rader | Kohorter (tiår): {sorted(df['birth_decade'].unique())}")
df.head()

## 1. Kaplan-Meier-inspirert overlevelseskurve per fødselskohort

Survival rate = andelen av kohorten som fortsatt er i live.

In [ ]:
# Aggreger over kjønn for total overlevelse per kohort
cohort_total = (
    df.groupby("birth_decade")
    .agg(cohort_size=("cohort_size", "sum"),
         alive_count=("alive_count", "sum"))
    .reset_index()
)
cohort_total["survival_rate"] = cohort_total["alive_count"] / cohort_total["cohort_size"].replace(0, np.nan) * 100
cohort_total = cohort_total.sort_values("birth_decade")

fig, ax = plt.subplots(figsize=(10, 5))
ax.step(cohort_total["birth_decade"], cohort_total["survival_rate"],
        where="post", linewidth=2.5, color="steelblue", label="Total")
ax.fill_between(cohort_total["birth_decade"], cohort_total["survival_rate"],
                step="post", alpha=0.1, color="steelblue")

# Per kjønn
cmap = {"M": "#3498db", "F": "#e91e8c", "X": "#8e44ad"}
for gender, grp in df.groupby("gender"):
    g = grp.sort_values("birth_decade")
    ax.step(g["birth_decade"], g["survival_rate_pct"],
            where="post", linewidth=1.2, linestyle="--",
            color=cmap.get(gender, "gray"), alpha=0.6, label=f"Kjønn: {gender}")

ax.yaxis.set_major_formatter(mticker.PercentFormatter())
ax.set_xlabel("Fødselsdekade")
ax.set_ylabel("Overlevelsesrate")
ax.set_title("Overlevelseskurve per fødselskohort (Kaplan-Meier-inspirert)", fontweight="bold")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 2. Giftermålsrate og skilsmisserate per kohort

In [ ]:
cohort_m = (
    df.groupby("birth_decade")
    .agg(cohort_size=("cohort_size", "sum"),
         married=("married_count", "sum"),
         divorced=("divorced_count", "sum"))
    .reset_index()
)
cohort_m["married_rate"]  = cohort_m["married"]  / cohort_m["cohort_size"].replace(0, np.nan) * 100
cohort_m["divorced_rate"] = cohort_m["divorced"] / cohort_m["cohort_size"].replace(0, np.nan) * 100
cohort_m = cohort_m.sort_values("birth_decade")

x     = np.arange(len(cohort_m))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - width/2, cohort_m["married_rate"],  width, label="Gift",     color="#3498db")
ax.bar(x + width/2, cohort_m["divorced_rate"], width, label="Skilt",    color="#e67e22")
ax.set_xticks(x)
ax.set_xticklabels([f"{int(d)}-tallet" for d in cohort_m["birth_decade"]], rotation=30, ha="right")
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
ax.set_ylabel("Andel av kohort")
ax.set_title("Giftermålsrate og skilsmisserate per fødselskohort", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()

## 3. Kohort-størrelse og kjønnsfordeling

In [ ]:
pivot = df.pivot_table(index="birth_decade", columns="gender", values="cohort_size", aggfunc="sum").fillna(0)
pivot = pivot.sort_index()

fig, ax = plt.subplots(figsize=(10, 4))
bottom = np.zeros(len(pivot))
colors_g = {"M": "#3498db", "F": "#e91e8c", "X": "#8e44ad"}

for gender in pivot.columns:
    vals = pivot[gender].values
    ax.bar(pivot.index, vals, bottom=bottom,
           label=f"Kjønn: {gender}", color=colors_g.get(gender, "gray"), width=7)
    bottom += vals

ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
ax.set_xlabel("Fødselsdekade")
ax.set_ylabel("Kohort-størrelse")
ax.set_title("Kohort-størrelse per fødselsdekade og kjønn", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()

print("\nSamlet statistikk per kohort:")
cohort_total[["birth_decade", "cohort_size", "alive_count", "survival_rate"]].to_string(index=False)

## 4. Gjennomsnittlig antall kommuneflyttinger per kohort

In [ ]:
mobility = (
    df.groupby("birth_decade")["avg_municipality_changes"]
    .mean()
    .reset_index()
    .sort_values("birth_decade")
)

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(mobility["birth_decade"], mobility["avg_municipality_changes"],
       color="#9b59b6", width=7, edgecolor="white")
ax.set_xlabel("Fødselsdekade")
ax.set_ylabel("Gj.snitt kommuneflyttinger")
ax.set_title("Geografisk mobilitet per fødselskohort", fontweight="bold")
for bar in ax.patches:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f"{bar.get_height():.2f}", ha="center", va="bottom", fontsize=8)
plt.tight_layout()
plt.show()